# Compounds against co-expression modules

Which genes of the tumour co-expression network does each compound move? The
modules found in `01_coexpression_modules.ipynb` are grouped, in Cytoscape,
into two large clusters (**red-purple** and **green-grey**); this notebook
intersects each compound's deregulated genes with those clusters.

**Produces** Supplementary Tables S18-S26
(`../../results/tables/wgcna/`).

⚠️ **Manual step upstream**: `TCGA_Network_0_13_full_table_attributes.csv` is
a Cytoscape node-attribute export. The two clusters were delimited visually in
Cytoscape from the edge list, not by an algorithm in this repository, so that
file cannot be regenerated by running the notebooks. It is included in
`../../results/tables/wgcna/`.

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "..")
from common import dose_window, g_scores, gene_ids_to_symbols

TABLES = "../../results/tables/wgcna"
DATA = "../../data/external"

Z_CUTOFF = 2
TOP_DOSE = "10 uM"   # per-compound gene lists are taken at the highest dose

## 1. Compound signatures

Same two classes as elsewhere: MC (mammary carcinogens) and SMC (suspected,
listed under `ED_merge.csv`), made disjoint by removing from SMC any compound
already classified as MC.

In [2]:
gene_info = pd.read_csv("../../data/lincs-metadata/geneinfo_beta.txt", sep="\t")
l1000 = pd.read_csv(f"{DATA}/L1000/L1000_MCF7_l_bi_bestTAS.csv")
l1000 = l1000[l1000["time"] == "24 h"]

mc_meta = (
    pd.read_csv("../../data/lincs-metadata/MC_merge.csv")
    .rename(columns={"inchi_key": "inchikey"})
    .drop_duplicates(subset=["inchikey"])
)
smc_meta = pd.read_csv("../../data/lincs-metadata/ED_merge.csv").rename(
    columns={"inchi_key": "inchikey"}
)
smc_meta = smc_meta[~smc_meta["inchikey"].isin(mc_meta["inchikey"])].drop_duplicates(
    subset=["inchikey"]
)

metadata = {"MC": mc_meta, "SMC": smc_meta}
signatures = {
    name: l1000[l1000["inchikey"].isin(meta["inchikey"])]
    for name, meta in metadata.items()
}
for name, df in signatures.items():
    print(f"{name:3s}: {len(df)} signatures, {df['inchikey'].nunique()} compounds")

MC : 220 signatures, 34 compounds
SMC: 840 signatures, 127 compounds


## 2. Class-level gene lists

The consistency score of `common.g_scores` applied to the dose window used
throughout the study, giving the genes each class moves as a whole.

In [3]:
for name, df in signatures.items():
    doses = dose_window(df["dose"].unique())
    data = df[df["dose"].isin(doses)].iloc[:, 7:]
    prefix = "mc" if name == "MC" else "ed"

    for direction in ("up", "down"):
        ids, scores, _ = g_scores(data, z_cutoff=Z_CUTOFF, direction=direction,
                                  return_scores=True)
        table = pd.DataFrame({
            "gene_symbol": gene_ids_to_symbols(gene_info, ids),
            f"{prefix}_{direction}_g_score": scores.values,
            f"{name}_{direction}reg": True,
        })
        table.to_csv(f"{TABLES}/{name}_{direction}regulated_genes_lbi.csv", index=False)
        print(f"{name:3s} {direction:4s}: {len(table)} genes")

MC  up  : 449 genes


MC  down: 204 genes


SMC up  : 341 genes


SMC down: 167 genes


## 3. Deregulated genes per compound

At the highest dose, the genes each individual compound moves past the
z-score threshold.

In [4]:
def genes_per_compound(df, direction):
    """InChIKey -> symbols of the genes it moves at the top dose."""
    top = df[df["dose"] == TOP_DOSE]
    per_compound = {}
    for row in top.itertuples(index=False):
        values = pd.Series(row[7:], index=top.columns[7:])
        passing = values > Z_CUTOFF if direction == "up" else values < -Z_CUTOFF
        ids = [int(g) for g in values.index[passing]]
        per_compound[row.inchikey] = gene_ids_to_symbols(gene_info, ids)
    return per_compound


compound_genes = {
    (name, direction): genes_per_compound(df, direction)
    for name, df in signatures.items()
    for direction in ("up", "down")
}

## 4. Overlap with the network clusters (Tables S19-S26)

In [5]:
attributes = pd.read_csv(f"{TABLES}/TCGA_Network_0_13_full_table_attributes.csv")

# Cytoscape exports these flags as True / empty, so pandas reads them as object
# columns holding True and NaN. They have to be coerced before being used as a
# boolean mask, otherwise pandas refuses the mask outright.
FLAGS = [c for c in attributes.columns if c.endswith(("_cluster", "reg"))]
attributes[FLAGS] = attributes[FLAGS].fillna(False).astype(bool)
print(f"{len(attributes)} network nodes; "
      + ", ".join(f"{c}={attributes[c].sum()}" for c in FLAGS))

clusters = {
    "Red_Purple": attributes.loc[attributes["Red_Purple_cluster"], "shared name"],
    "Green_Grey": attributes.loc[attributes["Green_Grey_cluster"], "shared name"],
}

for cluster_name, cluster_genes in clusters.items():
    cluster_genes = set(cluster_genes)
    for (name, direction), per_compound in compound_genes.items():
        rows = []
        for inchikey, genes in per_compound.items():
            shared = sorted(set(genes) & cluster_genes)
            rows.append({"inchikey": inchikey, "common_genes": shared,
                         "num_common_genes": len(shared)})

        table = pd.DataFrame(rows)
        table["comp_name"] = table["inchikey"].map(
            metadata[name].set_index("inchikey")["cmap_name"]
        )
        table = table[["comp_name", "num_common_genes", "common_genes", "inchikey"]]
        table = table.sort_values("num_common_genes", ascending=False)
        table.to_csv(
            f"{TABLES}/comp_gene_tables/{cluster_name}_Cluster_{name}"
            f"_compounds_{direction}regulated_genes.csv",
            index=False,
        )
        print(f"{cluster_name:11s} {name:3s} {direction:4s}: "
              f"{(table['num_common_genes'] > 0).sum()} compounds with ≥1 gene "
              f"in the cluster")

703 network nodes; ED_downreg=36, ED_upreg=29, Green_Grey_cluster=312, MC_downreg=54, MC_upreg=31, Red_Purple_cluster=190
Red_Purple  MC  up  : 13 compounds with ≥1 gene in the cluster
Red_Purple  MC  down: 25 compounds with ≥1 gene in the cluster
Red_Purple  SMC up  : 51 compounds with ≥1 gene in the cluster
Red_Purple  SMC down: 96 compounds with ≥1 gene in the cluster
Green_Grey  MC  up  : 23 compounds with ≥1 gene in the cluster
Green_Grey  MC  down: 3 compounds with ≥1 gene in the cluster
Green_Grey  SMC up  : 71 compounds with ≥1 gene in the cluster
Green_Grey  SMC down: 37 compounds with ≥1 gene in the cluster


## 5. Cluster genes per class (Table S18)

The reverse view: which genes of each cluster are moved by each class, taken
from the Cytoscape attributes directly.

In [6]:
for cluster_name, cluster_column in (("Red_Purple", "Red_Purple_cluster"),
                                     ("Green_Grey", "Green_Grey_cluster")):
    for name in ("MC", "ED"):
        for direction in ("up", "down"):
            selected = attributes[
                attributes[cluster_column] & attributes[f"{name}_{direction}reg"]
            ]
            (
                selected[["shared name"]]
                .rename(columns={"shared name": "gene_symbol"})
                .reset_index(drop=True)
                .to_csv(
                    f"{TABLES}/Network_013/{name}_{direction}reg_genes_in_"
                    f"{cluster_name}_cluster.csv",
                    index=False,
                )
            )
            print(f"{cluster_name:11s} {name:3s} {direction:4s}: {len(selected)} genes")

Red_Purple  MC  up  : 13 genes
Red_Purple  MC  down: 50 genes
Red_Purple  ED  up  : 18 genes
Red_Purple  ED  down: 33 genes
Green_Grey  MC  up  : 13 genes
Green_Grey  MC  down: 0 genes
Green_Grey  ED  up  : 6 genes
Green_Grey  ED  down: 1 genes
